# 🧠 Fine-Tuning T5-small on CNN/DailyMail with LoRA/PEFT
## SummarizationBench-LLM

**Runtime:** GPU (T4 recommended — Runtime → Change runtime type → T4 GPU)

This notebook reproduces the fine-tuning of T5-small using LoRA (Low-Rank Adaptation) on the CNN/DailyMail summarization dataset. After training, the LoRA adapter is saved and can be used with `evaluate.py` and `app.py`.

**What LoRA does:** Instead of updating all ~60M T5 parameters, LoRA injects small trainable matrices (~294K params, 0.485%) into the attention layers. This makes fine-tuning feasible on a free Colab T4 GPU.

---

### 📋 Notebook Structure
1. Install dependencies  
2. Load CNN/DailyMail dataset  
3. Set up T5 tokenizer and LoRA config  
4. Train with HuggingFace Trainer  
5. Visualize training loss  
6. Quick inference sanity check  
7. Download adapter to local machine  

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️ No GPU detected — switch runtime to T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# These are the only packages not in the default Colab environment:
%pip install -q transformers==4.38.0 datasets peft accelerate rouge-score nltk evaluate sentencepiece
print('✓ Dependencies installed')

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import json
import os
import re
import time
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback,
)
from peft import LoraConfig, TaskType, get_peft_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')

In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────────────────
# ⚙️ Adjust TRAIN_SIZE for speed vs. quality tradeoff:
#   5000  → ~15 min on T4  (good for testing the pipeline)
#   50000 → ~2 hrs on T4   (reasonable quality)
#   None  → full 287K      (~8 hrs on T4, best quality)

CONFIG = {
    'base_model':     'google/t5-small',
    'output_dir':     '/content/fine_tuned_t5_lora',
    'train_size':     5000,   # ← Change this
    'val_size':       500,
    'num_epochs':     3,
    'batch_size':     8,
    'grad_accum':     4,       # effective batch = 8 * 4 = 32
    'lr':             3e-4,
    'warmup_steps':   100,
    'max_input_len':  512,
    'max_target_len': 128,
    # LoRA config
    'lora_r':         8,
    'lora_alpha':     32,
    'lora_dropout':   0.1,
    'target_modules': ['q', 'v'],  # attention query and value projections
}
print('Configuration:', json.dumps(CONFIG, indent=2, default=str))

In [ ]:
# ── Cell 5: Load dataset ──────────────────────────────────────────────────────
print('Loading CNN/DailyMail dataset...')
raw_dataset = load_dataset('cnn_dailymail', '3.0.0')

if CONFIG['train_size']:
    raw_dataset['train'] = raw_dataset['train'].shuffle(seed=42).select(range(CONFIG['train_size']))
if CONFIG['val_size']:
    raw_dataset['validation'] = raw_dataset['validation'].shuffle(seed=42).select(range(CONFIG['val_size']))

print(f"Train: {len(raw_dataset['train'])} samples")
print(f"Val:   {len(raw_dataset['validation'])} samples")
print()

# Preview a sample
sample = raw_dataset['train'][0]
print('Article (first 300 chars):', sample['article'][:300])
print()
print('Highlights:', sample['highlights'])

In [ ]:
# ── Cell 6: Tokenize ──────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(CONFIG['base_model'])

def clean_text(text):
    text = re.sub(r'[^\x09\x0A\x0D\x20-\x7E\x80-\xFF]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_fn(examples):
    inputs  = ['summarize: ' + clean_text(doc) for doc in examples['article']]
    targets = [clean_text(ref) for ref in examples['highlights']]

    model_inputs = tokenizer(
        inputs, max_length=CONFIG['max_input_len'], truncation=True, padding='max_length'
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets, max_length=CONFIG['max_target_len'], truncation=True, padding='max_length'
        )

    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

tokenized = raw_dataset.map(
    preprocess_fn, batched=True, num_proc=2,
    remove_columns=['article', 'highlights', 'id'],
    desc='Tokenizing',
)
print('Tokenized splits:', {k: len(v) for k, v in tokenized.items()})

In [ ]:
# ── Cell 7: Build LoRA model ──────────────────────────────────────────────────
base_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['base_model'])

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['target_modules'],
    bias='none',
    inference_mode=False,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params: ~294,912 || all params: ~60,801,536 || trainable%: 0.485%

In [ ]:
# ── Cell 8: Loss logger callback ──────────────────────────────────────────────
class LossLogger(TrainerCallback):
    def __init__(self):
        self.history = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.history.append({'step': state.global_step, 'loss': round(logs['loss'], 6)})
    def on_train_end(self, args, state, control, **kwargs):
        with open('/content/losses.json', 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f'Loss history saved → /content/losses.json ({len(self.history)} steps)')

loss_logger = LossLogger()

In [ ]:
# ── Cell 9: Training arguments ────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['grad_accum'],
    learning_rate=CONFIG['lr'],
    warmup_steps=CONFIG['warmup_steps'],
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    fp16=True,                          # FP16 halves VRAM on T4
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=50,
    report_to='none',                   # disable wandb
    predict_with_generate=True,
    generation_max_length=CONFIG['max_target_len'],
    save_total_limit=2,
    dataloader_num_workers=0,
)
print('Training args ready.')

In [ ]:
# ── Cell 10: Train! ───────────────────────────────────────────────────────────
data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    label_pad_token_id=-100, pad_to_multiple_of=8
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[loss_logger, EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training...')
start = time.time()
trainer.train()
elapsed = (time.time() - start) / 60
print(f'\n✓ Training complete in {elapsed:.1f} minutes')

In [ ]:
# ── Cell 11: Save LoRA adapter ────────────────────────────────────────────────
# Only the adapter weights (~10 MB) are saved, NOT the full model
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f"✓ LoRA adapter saved → {CONFIG['output_dir']}")
print('Contents:', list(Path(CONFIG['output_dir']).iterdir()))

In [ ]:
# ── Cell 12: Plot training loss ───────────────────────────────────────────────
import matplotlib.pyplot as plt

with open('/content/losses.json') as f:
    history = json.load(f)

steps = [h['step'] for h in history]
losses = [h['loss'] for h in history]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, color='#7c6af7', linewidth=1.5, alpha=0.8)
ax.fill_between(steps, losses, alpha=0.15, color='#7c6af7')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('T5-small LoRA Training Loss')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_loss.png', dpi=150)
plt.show()
print('Loss curve saved → /content/training_loss.png')

In [ ]:
# ── Cell 13: Quick inference sanity check ─────────────────────────────────────
from peft import PeftModel

# Reload adapter (verifies the save was correct)
base = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['base_model'])
loaded_model = PeftModel.from_pretrained(base, CONFIG['output_dir'])
loaded_model = loaded_model.merge_and_unload()  # merge for fast inference
loaded_model.to(device).eval()

test_article = (
    "Scientists at MIT have developed a new biodegradable plastic that decomposes "
    "within 30 days in soil. The material, made from corn starch and a polymer "
    "compound, retains the same durability as conventional plastic during use. "
    "The team believes it could replace single-use plastics within five years."
)

tok_input = tokenizer(
    'summarize: ' + test_article,
    return_tensors='pt', max_length=512, truncation=True
).to(device)

with torch.no_grad():
    out = loaded_model.generate(
        **tok_input, max_new_tokens=100, num_beams=4, early_stopping=True
    )

summary = tokenizer.decode(out[0], skip_special_tokens=True)
print('Article:', test_article[:200], '...')
print()
print('T5-LoRA summary:', summary)

In [ ]:
# ── Cell 14: Download adapter to your local machine ───────────────────────────
# This will zip the adapter directory and trigger a download in Colab
import shutil
from google.colab import files

zip_path = '/content/fine_tuned_t5_lora.zip'
shutil.make_archive('/content/fine_tuned_t5_lora', 'zip', CONFIG['output_dir'])
files.download(zip_path)
print('Adapter zip downloaded. Extract it to ./fine_tuned_t5_lora/ in your project root.')
print('Then run: python src/evaluate.py --adapter_path ./fine_tuned_t5_lora')